# Surface Biomarkers Workflow

Example notebook for using the `surface_biomarkers` package with single-cell/snRNA-seq data.

Workflow:

1. load a count matrix;
2. filter surface genes or custom genes;
3. train one-vs-rest models;
4. generate metrics, ROC curves, confusion matrices and SHAP plots;
5. discover compact marker signatures;
6. prioritize signatures using surface and antibody databases;
7. design antibody panels.

## 1. Install In Editable Mode

Run this cell once. Restart the kernel if the package has just been installed.

In [ ]:
%pip install -e .

## 2. Imports

In [ ]:
from surface_biomarkers import (
    DiscoveryConfig,
    TrainingConfig,
    design_antibody_panel,
    discover_signatures,
    evaluate_signatures,
    load_counts_csv,
    prioritize_signature_validation,
    save_signature_validation_report,
    train_one_vs_rest,
)
from surface_biomarkers.plots import analyze_clusters, plot_panel_benchmark, plot_signature_accuracy, plot_shap_heatmap
from surface_biomarkers.signatures import (
    combined_shap_expression_matrix,
    mean_abs_shap_matrix,
)

import pandas as pd

## 3. Main Settings

Edit the paths and cluster labels for your dataset.

In [ ]:
COUNTS_CSV = "counts.csv"
TARGET_COLUMN = "celltype"
OUTPUT_DIR = "cluster_outputs"

clusters = ["0", "1", "2", "3", "4", "5"]

cluster_names = {
    "0": "Mac1",
    "1": "Mac2",
    "2": "Mac3",
    "3": "Mac4",
    "4": "Mac5",
    "5": "Mac6",
}

## 4. Load The Data

Use `gene_filter="surface"` to keep genes from the internal `db_surface.csv` database.

Options:

- `gene_filter="all"`: keep all genes;
- `gene_filter="surface"`: use the internal surface-gene database;
- `gene_filter="custom"`: use a user-defined gene list.

In [ ]:
dataset = load_counts_csv(
    COUNTS_CSV,
    target_column=TARGET_COLUMN,
    gene_filter="surface",
    encode_labels=False,
)

dataset.shape

### Optional: Use Custom Genes

In [ ]:
# genes_of_interest = ["PLXND1", "TLR1", "ABCA1", "CXADR", "FN1"]
# dataset = load_counts_csv(
#     COUNTS_CSV,
#     target_column=TARGET_COLUMN,
#     gene_filter="custom",
#     genes=genes_of_interest,
# )

## 5. Train One-vs-Rest Models

This trains one binary model per cluster: target cluster vs all other cells.

In [ ]:
training_config = TrainingConfig(
    resampling="SMOTE",
    cv=5,
    n_jobs=-1,
    verbose=1,
    grid_verbose=0,
    train_random_forest=True,
    train_lightgbm=True,
)

model_results = train_one_vs_rest(
    dataset,
    target_column=TARGET_COLUMN,
    output_dir=OUTPUT_DIR,
    config=training_config,
)

model_results

## 6. Generate Per-Cluster Metrics And Figures

This generates confusion matrices, ROC curves, individual SHAP summaries and one panel per cluster.

In [ ]:
metrics = analyze_clusters(
    base_dir=OUTPUT_DIR,
    clusters=clusters,
    model_name=["LGBM", "RF"],
    max_display=10,
    cluster_names=cluster_names,
    panel=True,
    show_panel=True,
)

metrics

## 7. Discover Compact Marker Signatures

Selects SHAP-prioritized genes, including positive and negative/exclusion candidates based on the SHAP-expression direction.

In [ ]:
discovery_config = DiscoveryConfig(
    model_name="LGBM",
    positive_genes=1,
    negative_genes=1,
    min_expression_prop=0.10,
    shap_corr_positive=0.15,
    shap_corr_negative=-0.10,
    top_n_heatmap=10,
)

signatures, gene_ranking, gene_exclusivity = discover_signatures(
    base_dir=OUTPUT_DIR,
    clusters=clusters,
    config=discovery_config,
)

signatures

In [ ]:
gene_ranking.head(20)

In [ ]:
gene_exclusivity.head(20)

## 8. SHAP Heatmaps

In [ ]:
shap_matrix = mean_abs_shap_matrix(OUTPUT_DIR, clusters, discovery_config)
combined_matrix = combined_shap_expression_matrix(OUTPUT_DIR, clusters, discovery_config)

plot_shap_heatmap(
    shap_matrix,
    f"{OUTPUT_DIR}/shap_top_genes_heatmap.svg",
    title="Top genes by mean absolute SHAP",
    cluster_names=cluster_names,
    show=True,
)

plot_shap_heatmap(
    combined_matrix,
    f"{OUTPUT_DIR}/shap_expression_heatmap.svg",
    title="SHAP x expression score",
    cluster_names=cluster_names,
    show=True,
)

## 9. Independent Accuracy Of Small Signatures

This function follows the original notebook logic: it sums signature genes in each `X_test.csv` block and computes accuracy using `score > threshold`.

In [ ]:
accuracy = evaluate_signatures(
    base_dir=OUTPUT_DIR,
    clusters=clusters,
    signatures=signatures,
    threshold=0.5,
    cluster_names=cluster_names,
)

accuracy

In [ ]:
plot_signature_accuracy(
    accuracy,
    f"{OUTPUT_DIR}/signature_accuracy.svg",
    cluster_names=cluster_names,
    show=True,
)

## 10. Translational Validation With Surface And Antibody Evidence

Uses the internal databases:

- `resources/db_surface.csv`
- `resources/proteinatlas.tsv`

Because the model was trained on snRNA-seq, these results are candidates for experimental validation, not validated protein markers.

In [ ]:
validation_priority = prioritize_signature_validation(
    ranking=gene_ranking,
    signatures=signatures,
    surface_db="internal",
    protein_atlas="internal",
    signature_only=True,
)

validation_priority.head(20)

In [ ]:
validation_priority = save_signature_validation_report(
    ranking=gene_ranking,
    signatures=signatures,
    surface_db="internal",
    protein_atlas="internal",
    output_path=f"{OUTPUT_DIR}/signature_validation_priority.csv",
    signature_only=True,
)

validation_priority.head()

## 11. Design Antibody Panels

The function below recommends marker combinations, not only individual genes.

It returns positive markers, negative/exclusion markers, backup markers and a minimal panel.

In [ ]:
panel_facs = design_antibody_panel(
    markers=validation_priority,
    application="FACS",
    max_positive_markers=2,
    max_negative_markers=1,
    max_backup_markers=2,
)

panel_facs

In [ ]:
panel_cite = design_antibody_panel(
    markers=validation_priority,
    application="CITE-seq",
)

panel_cite

## 12. Visual Panel Benchmark

Compares panel score, panel size, backup markers, redundancy penalty and application constraints by cluster/application.

In [ ]:
plot_panel_benchmark(
    panel_facs,
    f"{OUTPUT_DIR}/panel_benchmark_FACS.svg",
    cluster_names=cluster_names,
    show=True,
)

## 13. Save Main Tables

In [ ]:
gene_ranking.to_csv(f"{OUTPUT_DIR}/gene_ranking_by_cluster.csv", index=False)
gene_exclusivity.to_csv(f"{OUTPUT_DIR}/gene_exclusivity.csv", index=False)
accuracy.to_csv(f"{OUTPUT_DIR}/signature_accuracy.csv", index=False)
panel_facs.to_csv(f"{OUTPUT_DIR}/antibody_panel_FACS.csv", index=False)
panel_cite.to_csv(f"{OUTPUT_DIR}/antibody_panel_CITEseq.csv", index=False)

plot_panel_benchmark(
    panel_cite,
    f"{OUTPUT_DIR}/panel_benchmark_CITEseq.svg",
    cluster_names=cluster_names,
    show=False,
)